In [ ]:
import pandas as pd
import numpy as np
import gzip
import json
from datetime import datetime
from datetime import date
import plotly
import plotly.express as px 
from ast import literal_eval
import ast
import re
import os
import gzip
import json
import dask.dataframe as dd

 

In [2]:
import os
import gzip
import json

dir_path = '/Users/tpapka/Summer2025/xalt/data/test'

target_keys = ['libA', 'linkA', 'userT', 'userDT', 'envT']

records = []

for filename in os.listdir(dir_path):
    if filename.endswith('.gz'):
        file_path = os.path.join(dir_path, filename)
        with gzip.open(file_path, 'rt', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    try:
                        obj = json.loads(line)
                        extracted = {}

                        for key in target_keys:
                            if key in obj:
                                if key == "envT":
                                    env_data = obj.get("envT")
                                    if isinstance(env_data, dict):
                                        extracted["envT"] = {
                                            "LOADEDMODULES": env_data.get("LOADEDMODULES"),
                                            "LD_LIBRARY_PATH": env_data.get("LD_LIBRARY_PATH"),
                                            "CONDA_PROMPT_MODIFIER": env_data.get("CONDA_PROMPT_MODIFIER")
                                        }
                                else:
                                    extracted[key] = obj[key]

                        if extracted:
                            records.append(extracted)

                    except json.JSONDecodeError:
                        continue  
output_file = "extracted_records.json"
with open(output_file, "w") as out_f:
    json.dump(records, out_f, indent=4)

print(f"Wrote {len(records)} JSON records to: {output_file}")


Wrote 1246201 JSON records to: extracted_records.json


In [3]:
import json
import pandas as pd

input_file = "extracted_records.json"
chunk_size = 100_000
output_template = "normalized_chunk_{i}.parquet"

def stream_json_array_and_normalize(input_file, chunk_size):
    decoder = json.JSONDecoder()
    buffer = ""
    records = []
    chunk_idx = 0

    with open(input_file, "r") as f:
        f.read(1)  # skip opening '['

        while True:
            chunk = f.read(1024 * 1024)  # Read 1MB at a time
            if not chunk:
                break
            buffer += chunk
            while True:
                buffer = buffer.lstrip(", \n")
                try:
                    obj, idx = decoder.raw_decode(buffer)
                    records.append(obj)
                    buffer = buffer[idx:]
                except json.JSONDecodeError:
                    break  # Need more data

                if len(records) >= chunk_size:
                    flat = pd.json_normalize(records, sep="_")
                    flat.to_parquet(output_template.format(i=chunk_idx))
                    print(f"✅ Wrote chunk {chunk_idx} with {len(records)} rows")
                    records.clear()
                    chunk_idx += 1

    # Final chunk
    if records:
        flat = pd.json_normalize(records, sep="_")
        flat.to_parquet(output_template.format(i=chunk_idx))
        print(f"✅ Wrote final chunk {chunk_idx} with {len(records)} rows")

stream_json_array_and_normalize(input_file, chunk_size=100_000)


✅ Wrote chunk 0 with 100000 rows
✅ Wrote chunk 1 with 100000 rows
✅ Wrote chunk 2 with 100000 rows
✅ Wrote chunk 3 with 100000 rows
✅ Wrote chunk 4 with 100000 rows
✅ Wrote chunk 5 with 100000 rows
✅ Wrote chunk 6 with 100000 rows
✅ Wrote chunk 7 with 100000 rows
✅ Wrote chunk 8 with 100000 rows
✅ Wrote chunk 9 with 100000 rows
✅ Wrote chunk 10 with 100000 rows
✅ Wrote chunk 11 with 100000 rows
✅ Wrote final chunk 12 with 46201 rows


In [2]:
import pandas as pd
import glob

all_chunks = glob.glob("normalized_chunk_*.parquet")
#xalt = pd.concat([pd.read_parquet(p) for p in all_chunks], ignore_index=True)


In [3]:
#Dask implemnetaion 
xalt = dd.read_parquet(all_chunks)
# xalt.head(10)
xalt = xalt.repartition(partition_size='64MB')


In [4]:
polaris_data = pd.read_csv('/Users/tpapka/Summer2025/PyModuleSnooper/polaris_job_location_20250609.csv')


/var/folders/33/hsn1szm57t10kpxgfznwvncw0000gn/T/ipykernel_12349/616769568.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  polaris_data = pd.read_csv('/Users/tpapka/Summer2025/PyModuleSnooper/polaris_job_location_20250609.csv')


In [5]:
polaris_data['userT_job_id'] = polaris_data['JOB_NAME'].str.split('.').str[0]


In [6]:
xalt_polaris = xalt.merge(polaris_data, on = 'userT_job_id',how = 'left')

In [7]:
xalt_polaris

,libA,userT_syshost,userT_run_uuid,userT_exec_path,userT_exec_type,userT_cwd,userT_currentEpoch,userT_start_date,userT_user,userT_execModify,userT_scheduler,userT_account,userT_job_id,userT_queue,userT_submit_host,userDT_start_time,userDT_end_time,userDT_run_time,userDT_probability,userDT_num_tasks,userDT_num_gpus,userDT_exec_epoch,userDT_num_threads,userDT_num_cores,userDT_num_nodes,envT_LOADEDMODULES,envT_LD_LIBRARY_PATH,envT_CONDA_PROMPT_MODIFIER,userDT_Build_Epoch,END_DATE_ID,JOB_NAME,PROJECT_NAME,USERNAME,SCIENCE_FIELD_SHORT,QUEUED_TIMESTAMP,QUEUED_DATE_ID,START_TIMESTAMP,START_DATE_ID,END_TIMESTAMP,END_DATE_ID.1,NODES_USED,WALLTIME_SECONDS,RUNTIME_SECONDS,LOCATION
npartitions=39,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
,object,string,string,string,string,string,string,string,string,string,string,string,string,string,string,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,string,string,string,float64,int64,string,string,string,string,string,int64,string,int64,string,int64,float64,float64,float64,string
,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...


In [7]:
xalt_polaris_subset = xalt_polaris[['libA','userT_job_id','userDT_start_time','userDT_end_time','userDT_run_time','userDT_num_nodes','envT_CONDA_PROMPT_MODIFIER','PROJECT_NAME','SCIENCE_FIELD_SHORT','START_TIMESTAMP','END_TIMESTAMP','RUNTIME_SECONDS','LOCATION']]

In [9]:
# xalt = pd.read_json('extracted_records.json')

# xalt['JOB_NAME'] = xalt['userT'].apply(lambda x: x.get('job_id'))

In [10]:
#'libA', 'LOCATION', 'RUNTIME_SECONDS', 'WALLTIME_SECONDS', 'userDT_num_tasks', 'userDT_num_gpus', 'userDT_exec_epoch', 'userDT_num_threads', 'userDT_num_cores','userDT_num_nodes', 'userDT_run_time','envT_CONDA_PROMPT_MODIFIER','START_TIMESTAMP', 'END_TIMESTAMP','NODES_USED'
#xalt_polaris = xalt_polaris[['userT_job_id','libA', 'LOCATION', 'RUNTIME_SECONDS', 'WALLTIME_SECONDS', 'userDT_num_tasks', 'userDT_num_gpus', 'userDT_exec_epoch', 'userDT_num_threads', 'userDT_num_cores','userDT_num_nodes', 'userDT_run_time','envT_CONDA_PROMPT_MODIFIER','START_TIMESTAMP', 'END_TIMESTAMP','NODES_USED']]



In [8]:
xalt_explode = xalt_polaris_subset.explode('libA')


import numpy as np

def extract_path_from_array(x):
    if isinstance(x, list) and len(x) == 1 and isinstance(x[0], np.ndarray):
        array_val = x[0]
        return array_val[0] if len(array_val) > 0 else None
    elif isinstance(x, np.ndarray):
        return x[0] if len(x) > 0 else None
    return x

xalt_explode['libA'] = xalt_explode.map_partitions(
    lambda df: df['libA'].apply(extract_path_from_array),
    meta=('libA', 'object')
)






In [12]:
# xalt_explode['libA'] = xalt_explode['libA'].apply(
#     lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x
# )
xalt_explode

,libA,userT_job_id,userDT_start_time,userDT_end_time,userDT_run_time,userDT_num_nodes,envT_CONDA_PROMPT_MODIFIER,PROJECT_NAME,SCIENCE_FIELD_SHORT,START_TIMESTAMP,END_TIMESTAMP,RUNTIME_SECONDS,LOCATION
npartitions=39,,,,,,,,,,,,,
,object,string,float64,float64,float64,float64,string,string,string,string,string,float64,string
,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...


In [13]:
# xalt_explode['CONDA_PROMPT_MODIFIER'] = xalt_explode['envT'].apply(
#     lambda h: h.get('CONDA_PROMPT_MODIFIER') if isinstance(h, dict) else None
# )


In [9]:

# Step 1: Unwrap singleton lists in 'libA'
xalt_explode['libA'] = xalt_explode.map_partitions(
    lambda df: df['libA'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x),
    meta=('libA', 'object')
)

# Step 2: Classify 'libA' paths into multiple columns
def classify_libA_column(libA_series):
    def classify_path(path):
        py_lib = None
        standard_py = None
        non_python = None

        if isinstance(path, str):
            if 'python3.' in path:
                if '/site-packages/' in path:
                    match = re.search(r'/site-packages/([^/]+)/?', path)
                    if match:
                        py_lib = match.group(1)

                if 'lib-dynload' in path:
                    filename = path.split('/')[-1]
                    standard_py = filename.split('.')[0]
                else:
                    standard_py = 'no'
            else:
                non_python = path.split('/')[-1]
        return pd.Series([py_lib, standard_py, non_python], index=['PyLibraries', 'standard', 'non_python'])

    return libA_series.apply(classify_path)

# Apply classification
xalt_explode[['PyLibraries', 'standard', 'non_python']] = xalt_explode.map_partitions(
    lambda df: classify_libA_column(df['libA']),
    meta={'PyLibraries': 'object', 'standard': 'object', 'non_python': 'object'}
)


In [15]:
#xalt_explode.head(n=5, npartitions=1)


In [10]:
def from_home(path):
    from_home = None
    if isinstance(path, str):
        if '/home' in path:
            from_home = 'yes'
        else:
            from_home = 'no'
    return pd.Series([from_home], index=['from_home'])

# Apply with map_partitions, making sure to pass a Series and get a DataFrame back
def apply_from_home(series):
    return series.apply(from_home)

xalt_explode[['from_home']] = xalt_explode.map_partitions(
    lambda df: apply_from_home(df['libA']),
    meta={'from_home': 'object'}
)

In [11]:

def conda_classify(x):
    if x == '(base) ':
        return 'base'
    elif pd.notna(x):
        return 'extended'
    else:
        return None

xalt_explode['conda'] = xalt_explode.map_partitions(
    lambda df: df['envT_CONDA_PROMPT_MODIFIER'].apply(conda_classify),
    meta=('conda', 'object')
)


### Where the trouble begins

In [ ]:
xalt_explode.head(10)

In [18]:
xalt_explode['PyLibraries'] = xalt_explode['PyLibraries'].astype('category')


In [19]:
xalt_explode['conda'] = xalt_explode['conda'].astype('category')


In [20]:
conda_counts = xalt_explode.groupby('conda').size()
print(conda_counts)

Dask Series Structure:
npartitions=1
    int64
      ...
Dask Name: size, 27 expressions
Expr=Size(frame=Assign(frame=Assign(frame=Assign(frame=Assign(frame=Assign(frame=Assign(frame=Assign(frame=ExplodeFrame(frame=Merge(4678040)[['libA', 'userT_job_id', 'userDT_start_time', 'userDT_end_time', 'userDT_run_time', 'userDT_num_nodes', 'envT_CONDA_PROMPT_MODIFIER', 'PROJECT_NAME', 'SCIENCE_FIELD_SHORT', 'START_TIMESTAMP', 'END_TIMESTAMP', 'RUNTIME_SECONDS', 'LOCATION']], column=['libA'])))))))), observed=False)


/opt/anaconda3/lib/python3.12/site-packages/dask/dataframe/dask_expr/_groupby.py:1561: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  self._meta = self.obj._meta.groupby(


In [ ]:
conda_counts = conda_counts.compute()


In [ ]:
#xalt_explode.get_partition(0).memory_usage(deep=True).compute().sum() / 1e6  # in MB


In [ ]:
# ddf_sample = xalt_explode.partitions[:2]
# sample_counts = ddf_sample['PyLibraries'].value_counts().compute()


In [ ]:
#print(sample_counts).head(5)

In [ ]:
# sample = xalt_explode['PyLibraries'].sample(frac=0.01).compute()
# print(sample.nunique())

In [ ]:
#xalt_py_lib_count = xalt_explode['PyLibraries'].value_counts().compute()